# Relate entities of two domains that lie in one database

A resource is a database, so two domains whose data lie in one database share one resource, and it lists the entities of both. A relation between those entities is part of the model like any other, and the graph links the database to each entity it lists.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

In [ ]:
!pip install -q aoa-action-machine

In [2]:
import json
from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationOne, BaseCoreEntity, NoInverse
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.graph.node_graph_coordinator_factory import create_node_graph_coordinator
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources import BaseStorage

## 2. Declare the model

In [3]:
class StoreDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "store"
    description = "Store"


class SupportDomain(BaseDomain):
    """Group the support desk declarations."""

    name = "support"
    description = "Support"


@entity(description="Customer order", domain=StoreDomain)
class SalesOrderCoreEntity(BaseCoreEntity):
    """A customer's order."""

    id: str = Field(description="Order identifier")


@entity(description="Support ticket", domain=SupportDomain)
class SupportTicketCoreEntity(BaseCoreEntity):
    """A ticket about an order; the ticket and the order lie in the same database."""

    id: str = Field(description="Ticket identifier")
    order: Annotated[AssociationOne[SalesOrderCoreEntity], NoInverse()] = Field(description="Order")


@meta(description="The shop database")
@entities(
    SalesOrderCoreEntity,
    SupportTicketCoreEntity,
)
class ShopDatabaseStorage(BaseStorage):
    """One database keeps the store's orders and the support desk's tickets."""

## 3. Run and check the observation

In [4]:
SupportTicketCoreEntity.model_rebuild()

# Run the case.
graph = json.loads(create_node_graph_coordinator().to_json())
short = {node["id"]: node["label"] for node in graph["nodes"]}
for edge in graph["edges"]:
    if edge["type"] in ("entity_relation", "resource_entity"):
        print(edge["type"] + ":", short[edge["source_id"]], "->", short[edge["target_id"]])

resource_entity: ShopDatabaseStorage -> SalesOrderCoreEntity
resource_entity: ShopDatabaseStorage -> SupportTicketCoreEntity
entity_relation: SupportTicketCoreEntity -> SalesOrderCoreEntity


## 4. Explain the result

One database keeps the entities of two domains, so one resource lists both, and the relation between them is part of the model like any other.